In [1]:
import numpy as np
import pandas as pd

# 1. 模擬數據
np.random.seed(42)
time = np.arange(100)
# 正常微小波動（核心概念：標準差為 1 的雜訊）
core_noise = np.random.normal(0, 1, 100) 

# 基准線：前 50 點是 10，後 50 點換管柱變成 50
baseline = np.array([10]*50 + [50]*50)   

# 實際感測器讀到的數據（包含突變的 Drift）
sensor_data = baseline + core_noise

# 在第 75 點人為製造一個「真正的異常突刺」(+15)
sensor_data[75] += 15 

# 2. 模擬論文的「動態分解」（這裡用簡單的滑動平均代替）
df = pd.DataFrame({'raw': sensor_data})
df['drift'] = df['raw'].rolling(window=5, center=True).mean().bfill().ffill()
df['residual'] = df['raw'] - df['drift']

# 3. 看看結果
print(df.iloc[[48, 52, 75]]) # 觀察換管柱前後，以及異常發生時

          raw      drift   residual
48  10.343618  17.900229  -7.556611
52  49.323078  50.180951  -0.857873
75  65.821903  52.910968  12.910934


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
sns.set_theme(style='darkgrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
raw_data = pd.read_csv('../data/processed/mhd_ch4_ml_v2.csv')
raw_data.head(20)

/tmp/ipykernel_18577/2919345968.py:1: DtypeWarning: Columns (37) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv('../data/processed/mhd_ch4_ml_v2.csv')


,date,time,type,sample,standard,port,ht,tmod,tamb,lab_temp,pflow,psamp,ploop,pamb,CH4_rt,CH4_w,CH4_ht,CH4_area,CH4_skew,CH4_start_time,CH4_end_time,CH4_start_level,CH4_end_level,CH4_Rl_ht,CH4_Rl_a,CH4_Rl,CH4_norm_ht,CH4_norm_a,CH4_norm_w,CH4_R_ht,CH4_R_a,CH4_R,CH4_C_ht,CH4_C_a,CH4_C,CH4_std_stdev,CH4_std_rep,CH4_Cstd,CH4_flag_ht,CH4_flag_a,CH4_flag,CH4_flag_p,CH4_flag_ht_bin,CH4_flag_a_bin,label1,datetime_str,datetime,is_ht_999,is_area_999,is_rt_zero,is_w_zero,duration,is_duration_spike,baseline_drift,ht_area_ratio,last_std_time,seconds_since_last_std,type_std,type_air,type_tank,type_blank,type_cal,type_other,last_cal_time,seconds_since_last_cal,year,month,day,hour,minute,seconds,psamp_roll_mean_10,pamb_roll_mean_10,pflow_roll_mean_10,tmod_roll_mean_10,psamp_roll_mean_100,pamb_roll_mean_100,pflow_roll_mean_100,tmod_roll_mean_100,psamp_roll_std_10,pamb_roll_std_10,pflow_roll_std_10,CH4_rt_roll_std_10,psamp_roll_std_100,pamb_roll_std_100,pflow_roll_std_100,CH4_rt_roll_std_100,psamp_diff_1,pflow_diff_1,CH4_area_diff_1,psamp_lag_1,pflow_lag_1
0,940216,142900,air,10m,G-024,3,10.0,40.00,32.27,NaN,841.9,761.10,NaN,761.1,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,F,F,,,1,1,1,940216142900,1994-02-16 14:29:00,False,False,True,True,0.0,False,0.0,NaN,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,14,29,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,940216,144900,air,10m,G-024,3,10.0,40.00,32.48,NaN,840.4,761.07,NaN,761.2,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,F,F,,,1,1,1,940216144900,1994-02-16 14:49:00,False,False,True,True,0.0,False,0.0,NaN,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,14,49,0,761.100000,761.100000,841.900000,40.000000,761.100000,761.100000,841.900000,40.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,761.10,841.9
2,940216,150900,air,10m,G-024,3,10.0,40.01,32.75,NaN,839.4,760.99,NaN,760.9,109.8,2.60,190908.0,496440.0,0.63,102.0,135.4,10063.0,10029.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,F,F,,,1,1,1,940216150900,1994-02-16 15:09:00,False,False,False,False,33.4,False,-34.0,0.384554,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,15,9,0,761.085000,761.150000,841.150000,40.000000,761.085000,761.150000,841.150000,40.000000,0.021213,0.070711,1.060660,0.000000,0.021213,0.070711,1.060660,0.000000,-0.03,-1.5,0.0,761.07,840.4
3,940216,152900,air,10m,G-024,3,10.0,40.00,32.92,NaN,839.6,760.96,NaN,760.9,94.8,6.85,151667.0,1065982.0,1.02,86.2,129.0,99999.0,99999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,,,,,0,0,0,940216152900,1994-02-16 15:29:00,False,False,False,False,42.8,False,0.0,0.142279,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,15,29,0,761.053333,761.066667,840.566667,40.003333,761.053333,761.066667,840.566667,40.003333,0.056862,0.152753,1.258306,63.393060,0.056862,0.152753,1.258306,63.393060,-0.08,-1.0,496440.0,760.99,839.4
4,940216,180000,air,10m,G-024,3,10.0,40.00,33.31,NaN,845.4,760.72,NaN,760.6,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,F,F,,,1,1,1,940216180000,1994-02-16 18:00:00,False,False,True,True,0.0,False,0.0,NaN,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,18,0,0,761.030000,761.025000,840.325000,40.002500,761.030000,761.025000,840.325000,40.002500,0.065828,0.150000,1.135415,59.379542,0.065828,0.150000,1.135415,59.379542,-0.03,0.2,569542.0,760.96,839.6
5,940216,182000,air,10m,G-024,3,10.0,40.01,33.10,NaN,846.2,760.60,NaN,760.7,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,F,F,,,1,1,1,940216182000,1994-02-16 18:20:00,False,False,True,True,0.0,False,0.0,NaN,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,18,20,0,760.968000,760.940000,841.340000,40.002000,760.968000,760.940000,841.340000,40.002000,0.149900,0.230217,2.473459,56.282431,0.149900,0.230217,2.473459,56.282431,-0.24,5.8,-1065982.0,760.72,845.4
6,940216,184000,air,10m,G-024,3,

In [4]:
data = raw_data.copy()
data = data.drop(data[data['year']==2026].index)
data.tail()

,date,time,type,sample,standard,port,ht,tmod,tamb,lab_temp,pflow,psamp,ploop,pamb,CH4_rt,CH4_w,CH4_ht,CH4_area,CH4_skew,CH4_start_time,CH4_end_time,CH4_start_level,CH4_end_level,CH4_Rl_ht,CH4_Rl_a,CH4_Rl,CH4_norm_ht,CH4_norm_a,CH4_norm_w,CH4_R_ht,CH4_R_a,CH4_R,CH4_C_ht,CH4_C_a,CH4_C,CH4_std_stdev,CH4_std_rep,CH4_Cstd,CH4_flag_ht,CH4_flag_a,CH4_flag,CH4_flag_p,CH4_flag_ht_bin,CH4_flag_a_bin,label1,datetime_str,datetime,is_ht_999,is_area_999,is_rt_zero,is_w_zero,duration,is_duration_spike,baseline_drift,ht_area_ratio,last_std_time,seconds_since_last_std,type_std,type_air,type_tank,type_blank,type_cal,type_other,last_cal_time,seconds_since_last_cal,year,month,day,hour,minute,seconds,psamp_roll_mean_10,pamb_roll_mean_10,pflow_roll_mean_10,tmod_roll_mean_10,psamp_roll_mean_100,pamb_roll_mean_100,pflow_roll_mean_100,tmod_roll_mean_100,psamp_roll_std_10,pamb_roll_std_10,pflow_roll_std_10,CH4_rt_roll_std_10,psamp_roll_std_100,pamb_roll_std_100,pflow_roll_std_100,CH4_rt_roll_std_100,psamp_diff_1,pflow_diff_1,CH4_area_diff_1,psamp_lag_1,pflow_lag_1
808206,251231,223800,std,J-286,J-286,3,0.0,40.0,21.01,NaN,817.8,765.45,NaN,765.7,94.0,7.83,170066.0,1489856.0,1.16,80.8,133.4,23823.0,24073.0,1.00099,1.00067,1.00067,1.0010,1.0007,0.9997,1.00099,1.00067,1.00067,2017.92,2017.27,2017.27,2.406,NaN,2015.923,,,A,,0,0,0,251231223800,2025-12-31 22:38:00,False,False,False,False,52.6,False,250.0,0.114149,2025-12-31 21:57:00,2460.0,1,0,0,0,0,0,2025-12-28 06:16:00,318120.0,2025,12,31,22,38,0,766.407,766.33,816.44,40.0,772.0808,772.056,822.436,40.0,0.583306,0.586989,2.761320,0.10328,2.832881,2.881439,3.691583,0.094516,-0.47,-4.6,34189.0,765.37,813.7
808207,251231,225800,air,9m,J-286,1,9.0,40.0,20.74,NaN,813.2,765.28,NaN,765.2,94.0,7.85,171821.0,1508117.0,1.16,80.8,133.4,23750.0,24058.0,1.01178,1.01300,1.01300,1.0118,1.0130,1.0014,1.01178,1.01300,1.01300,2039.67,2042.12,2042.12,2.396,2.427,2015.923,,,A,,0,0,0,251231225800,2025-12-31 22:58:00,False,False,False,False,52.6,False,308.0,0.113931,2025-12-31 21:57:00,3660.0,0,1,0,0,0,0,2025-12-28 06:16:00,319320.0,2025,12,31,22,58,0,766.222,766.18,816.24,40.0,771.9890,771.966,822.381,40.0,0.561581,0.528730,2.555691,0.10328,2.897443,2.938020,3.719446,0.095219,0.08,4.1,-34481.0,765.45,817.8
808208,251231,231800,std,J-286,J-286,3,0.0,40.0,20.88,NaN,817.5,765.08,NaN,764.9,94.2,7.84,169569.0,1487622.0,1.06,80.8,133.6,23740.0,24030.0,0.99744,0.99876,0.99876,0.9973,0.9986,1.0015,0.99744,0.99876,0.99876,2010.76,2013.42,2013.42,2.413,NaN,2015.923,,,A,,0,0,0,251231231800,2025-12-31 23:18:00,False,False,False,False,52.8,False,290.0,0.113987,2025-12-31 22:38:00,2400.0,1,0,0,0,0,0,2025-12-28 06:16:00,320520.0,2025,12,31,23,18,0,766.039,766.00,816.19,40.0,771.8964,771.871,822.240,40.0,0.537720,0.524934,2.615106,0.10328,2.962336,3.001626,3.797527,0.095874,-0.17,-4.6,18261.0,765.28,813.2
808209,251231,233800,air,9m,J-286,1,9.0,40.0,20.79,NaN,813.3,765.05,NaN,765.0,94.2,7.86,170281.0,1496142.0,1.07,80.8,133.6,23756.0,24015.0,1.00289,1.00503,1.00503,1.0030,1.0051,1.0021,1.00289,1.00503,1.00503,2021.75,2026.06,2026.06,2.401,2.413,2015.923,,,A,,0,0,0,251231233800,2025-12-31 23:38:00,False,False,False,False,52.8,False,259.0,0.113813,2025-12-31 22:38:00,3600.0,0,1,0,0,0,0,2025-12-28 06:16:00,321720.0,2025,12,31,23,38,0,765.872,765.82,816.02,40.0,771.8028,771.775,822.179,40.0,0.551519,0.565292,2.447584,0.10328,3.028294,3.069445,3.824358,0.095874,-0.20,4.3,-20495.0,765.08,817.5
808210,251231,235800,std,J-286,J-286,3,0.0,NaN,NaN,NaN,817.4,764.90,NaN,NaN,94.2,7.83,169985.0,1489453.0,1.09,80.8,133.4,23691.0,24015.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.415,NaN,2015.923,,,A,,0,0,0,251231235800,2025-12-31 23:58:00,False,False,False,False,52.6,False,324.0,0.114126,2025-12-31 23:18:00,2400.0,1,0,0,0,0,0,2025-12-28 06:16:00,322920.0,2025,12,31,23,58,0,765.710,765.66,815.82,40.0,771.7077,771.680,822.040,40.0,0.528520,0.546097,2.590495,0.10328,3.089538,3.130656,3.892028,0.095874,-0.03,-4.2,8520.0,765.05,813.3


In [5]:
data[data['type_cal'] == 1].head(20)

,date,time,type,sample,standard,port,ht,tmod,tamb,lab_temp,pflow,psamp,ploop,pamb,CH4_rt,CH4_w,CH4_ht,CH4_area,CH4_skew,CH4_start_time,CH4_end_time,CH4_start_level,CH4_end_level,CH4_Rl_ht,CH4_Rl_a,CH4_Rl,CH4_norm_ht,CH4_norm_a,CH4_norm_w,CH4_R_ht,CH4_R_a,CH4_R,CH4_C_ht,CH4_C_a,CH4_C,CH4_std_stdev,CH4_std_rep,CH4_Cstd,CH4_flag_ht,CH4_flag_a,CH4_flag,CH4_flag_p,CH4_flag_ht_bin,CH4_flag_a_bin,label1,datetime_str,datetime,is_ht_999,is_area_999,is_rt_zero,is_w_zero,duration,is_duration_spike,baseline_drift,ht_area_ratio,last_std_time,seconds_since_last_std,type_std,type_air,type_tank,type_blank,type_cal,type_other,last_cal_time,seconds_since_last_cal,year,month,day,hour,minute,seconds,psamp_roll_mean_10,pamb_roll_mean_10,pflow_roll_mean_10,tmod_roll_mean_10,psamp_roll_mean_100,pamb_roll_mean_100,pflow_roll_mean_100,tmod_roll_mean_100,psamp_roll_std_10,pamb_roll_std_10,pflow_roll_std_10,CH4_rt_roll_std_10,psamp_roll_std_100,pamb_roll_std_100,pflow_roll_std_100,CH4_rt_roll_std_100,psamp_diff_1,pflow_diff_1,CH4_area_diff_1,psamp_lag_1,pflow_lag_1
169,940219,50700,cal560,G-024,G-024,1,0.0,40.00,23.67,NaN,848.0,561.34,NaN,752.6,94.4,6.63,122237.0,877875.0,1.03,83.0,127.4,31579.0,31373.0,1.03722,1.00072,1.03722,0.7734,0.7462,0.9623,1.03722,1.00072,1.03722,1755.47,1693.70,1692.49,3.165,NaN,1692.489,,,,,0,0,0,940219050700,1994-02-19 05:07:00,False,False,False,False,44.4,False,-206.0,0.139242,1994-02-19 04:06:00,3660.0,0,0,0,0,1,0,NaN,NaN,1994,2,19,5,7,0,646.046,754.19,841.22,40.000,721.9993,754.459,840.993,40.000100,228.042193,0.902404,8.552297,0.710555,128.664495,3.133788,7.496481,0.508132,-0.34,15.8,-103599.0,752.90,848.0
173,940219,62700,cal660,G-024,G-024,1,0.0,40.01,23.65,NaN,847.6,659.79,NaN,751.9,94.6,6.75,140908.0,1030847.0,1.10,83.2,128.0,31408.0,31196.0,1.01558,1.00040,1.01558,0.8910,0.8777,0.9831,1.01558,1.00040,1.01558,1718.86,1693.17,1692.49,3.185,NaN,1692.489,,,,,0,0,0,940219062700,1994-02-19 06:27:00,False,False,False,False,44.8,False,-212.0,0.136691,1994-02-19 05:27:00,3600.0,0,0,0,0,1,0,1994-02-19 05:07:00,4800.0,1994,2,19,6,27,0,680.100,753.15,842.21,40.001,720.2088,754.583,841.417,40.000300,174.448026,0.754615,8.030975,0.482586,129.652293,2.986943,7.390371,0.505369,-0.21,15.0,-98263.0,752.19,848.0
177,940219,74700,cal860,G-024,G-024,1,0.0,39.99,23.44,NaN,847.7,858.82,NaN,751.6,95.2,7.07,175756.0,1342229.0,1.08,83.2,129.6,31177.0,31035.0,0.97650,1.00179,0.97650,1.1159,1.1448,1.0266,0.97650,1.00179,0.97650,1652.72,1695.52,1692.49,3.130,NaN,1692.489,,,,,0,0,0,940219074700,1994-02-19 07:47:00,False,False,False,False,46.4,False,-142.0,0.130943,1994-02-19 06:47:00,3600.0,0,0,0,0,1,0,1994-02-19 06:27:00,4800.0,1994,2,19,7,47,0,723.966,752.28,843.04,40.001,719.3836,754.679,841.703,40.000200,64.116355,0.524510,7.839529,0.402216,129.775334,2.851379,7.296207,0.512356,-0.08,17.6,-92070.0,751.60,847.6
181,940219,90700,cal960,G-024,G-024,1,0.0,40.01,23.55,NaN,847.3,958.55,NaN,751.1,95.2,7.28,191472.0,1497488.0,1.01,82.8,129.8,31034.0,30836.0,0.95351,1.00127,0.95351,1.2169,1.2778,1.0561,0.95351,1.00127,0.95351,1613.80,1694.64,1692.49,3.168,NaN,1692.489,,,,,0,0,0,940219090700,1994-02-19 09:07:00,False,False,False,False,47.0,False,-198.0,0.127862,1994-02-19 08:07:00,3600.0,0,0,0,0,1,0,1994-02-19 07:47:00,4800.0,1994,2,19,9,7,0,753.232,751.69,842.69,40.002,720.5308,754.752,841.963,40.000300,47.022329,0.417532,7.874212,0.370585,130.502579,2.738077,7.225623,0.512703,-0.12,16.3,-88903.0,751.18,847.3
185,940219,102700,cal1060,G-024,G-024,1,0.0,40.01,23.16,NaN,846.8,1058.30,NaN,751.2,95.2,7.51,205489.0,1651923.0,1.00,82.6,130.2,30872.0,30727.0,0.92736,1.00177,0.92736,1.3073,1.4122,1.0901,0.92736,1.00177,0.92736,1569.54,1695.49,1692.49,3.180,NaN,1692.489,,,,,0,0,0,940219102700,1994-02-19 10:27:00,False,False,False,False,47.6,False,-145.0,0.124394,1994-02-19 09:27:00,3600.0,0,0,0,0,1,0,1994-02-19 09:07:00,4800.0,1994,2,19,10,27,0,782.750,751.27,841.92,40.002,722.6547,754.798,842.177,40.000500,70.413667,0.279086,8.243624,0.386437,132.635435,2.663

In [6]:
(data['type_cal'] == 1).sum()

np.int64(14027)

In [7]:
years = data.loc[data['type_cal'] == 1, 'year'].unique()

In [8]:
for i in years:
    one_year = data[data['year']==i]
    print(f"{i}",one_year.loc[data['type_cal'] == 1, 'month'].unique())

1994 [ 2  3  4  5  6  7  8  9 10 11 12]
1995 [ 1  2  3  4  5  6  7  8  9 10 11 12]
1996 [ 1  2  3  4  5  6  7  8  9 10 11 12]
1997 [ 1  2  3  4  5  6  7  8  9 10 11 12]
1998 [ 1  2  3  4  5  6  7  8  9 10 11 12]
1999 [ 1  2  3  4  5  6  7  8  9 10 11 12]
2000 [ 1  2  3  4  5  6  7  8  9 10 11 12]
2001 [ 1  2  3  4  5  6  7  8  9 10 11 12]
2002 [ 1  2  3  4  5  6  7  8  9 10 11 12]
2003 [ 1  2  3  4  5  6  7  8  9 10 11 12]
2004 [ 1  2  3  4  5  6  7  8  9 10 11 12]
2005 [ 1  2  3  4  5  6  7  8  9 10 11 12]
2006 [ 1  2  3  4  5  6  7  8  9 10 11 12]
2007 [ 1  2  3  4  5  6  7  8  9 10 11 12]
2008 [ 1  2  3  4  5  6  7  8  9 10 11 12]
2009 [1 2 3 4 5]
2010 [2]
2011 [6]
2016 [2]
2018 [ 4 11 12]
2019 [ 1  3  6  7  8 10 12]
2020 [ 4  5  6  7  9 10]
2021 [ 7  8  9 10 12]
2022 [ 1  2  3  4  5  6  7  8  9 10 11 12]
2023 [ 1  2  3  4  5  8  9 10 12]
2024 [1]
2025 [10 11 12]


In [9]:
std_data = data[data['type_std']==1]
((std_data['year']==1994)&(std_data['month']==2)&(std_data['day']==18)).sum()

np.int64(36)

In [10]:
((std_data['year']==2005)&(std_data['month']==2)&(std_data['day']==18)).sum()

np.int64(36)

one day -> 36 data points

In [11]:
feature_cols = [
    #'sample'	,'standard',	
    #'port'	,'ht',
    'tmod'	,'tamb',
    'pflow', 'psamp',
    'pamb'	,
    'CH4_rt',	'CH4_w',	'CH4_ht'	,
    'CH4_area',	'CH4_skew'	,
    'CH4_start_time',	'CH4_end_time'	,'CH4_start_level',	'CH4_end_level',
    #'CH4_Rl_ht',	'CH4_Rl_a',	'CH4_Rl',	
    #'CH4_norm_ht','CH4_norm_a',	'CH4_norm_w',
    #'CH4_R_ht','CH4_R_a', 'CH4_R'	,'
    #'CH4_C_ht'	,'CH4_C_a',	'CH4_C',	
    # std
    #'CH4_std_rep', 'CH4_std_rep'
    #'CH4_Cstd', 
    #'is_ht_999', 'is_area_999', 'is_rt_zero', 'is_w_zero',
    'duration', 
    #'is_duration_spike',
    #'baseline_drift', 
    'ht_area_ratio',
    #'type_air', 'type_blank', 'type_cal', 'type_other', 'type_std', 'type_tank',
    #'year', 'month', 'day', 'hour', 'minute', 'seconds'
]

In [12]:
import matplotlib.pyplot as plt
air_data = data[data['type_air']==1]
air_data = air_data.sort_values('datetime')
df = air_data.copy()
window_list = {
    '0.5_days': 18,
    '1_days': 36,
    '7_days': 36 * 7,
    '14_days': 36 * 14,
    '21_days': 36 * 21,
    '30_days': 36*30,
    '60_days': 36 * 60,
    '90_days': 36 * 90,
    '120_days': 36 * 120,
    '150_days': 36 * 150,
    '300_days': 36 * 300,
    '360_days': 36 * 360,
    '720_days':36*720
}




#for y in years:

for f in feature_cols:
    results = {}
    print(f"Feature  {f}\n")
    for n, w in window_list.items():
        rolling_drift= df[f].rolling(window=w, min_periods=1, center=True).mean()

        residual = df[f] - rolling_drift

        #mad = np.median(np.abs(residual - np.median(residual)))
        rmse= np.sqrt(np.mean(residual ** 2))
        results[n] = {
            'rmse':rmse,
            'residual': residual
        }

        print(f"Window Size:{n:11s} | residualmad score: {rmse:.10f}")

    best_window = min(results, key=lambda k: results[k]['rmse'])
    print(f"\n best roling window size: {best_window}\n")



Feature  tmod

Window Size:0.5_days    | residualmad score: 0.0578246564
Window Size:1_days      | residualmad score: 0.0640094066
Window Size:7_days      | residualmad score: 0.0787605668
Window Size:14_days     | residualmad score: 0.0853629174
Window Size:21_days     | residualmad score: 0.0893692112
Window Size:30_days     | residualmad score: 0.0930073140
Window Size:60_days     | residualmad score: 0.1061962638
Window Size:90_days     | residualmad score: 0.1121623460
Window Size:120_days    | residualmad score: 0.1148750529
Window Size:150_days    | residualmad score: 0.1164239985
Window Size:300_days    | residualmad score: 0.1193482720
Window Size:360_days    | residualmad score: 0.1198606074
Window Size:720_days    | residualmad score: 0.1208006240

 best roling window size: 0.5_days

Feature  tamb

Window Size:0.5_days    | residualmad score: 0.4215546781
Window Size:1_days      | residualmad score: 0.4913266046
Window Size:7_days      | residualmad score: 0.7803419524
Windo

In [13]:
import matplotlib.pyplot as plt

std_data = std_data.sort_values('datetime')
df = std_data.copy()
window_list = {
    '0.5_days': 18,
    '1_days': 36,
    '7_days': 36 * 7,
    '14_days': 36 *14,
    '21_days': 36 * 21,
    '30_days': 36*30,
    '60_days': 36 * 60,
    '90_days': 36 * 90,
    '120_days': 36 * 120,
    '150_days': 36 * 150,
    '300_days': 36 * 300,
    '360_days': 36 * 360,
    '720_days':36*720
}




#for y in years:

for f in feature_cols:
    results = {}
    print(f"Feature  {f}\n")
    for n, w in window_list.items():
        rolling_drift= df[f].rolling(window=w, min_periods=1, center=True).mean()

        residual = df[f] - rolling_drift

        #mad = np.median(np.abs(residual - np.median(residual)))
        rmse= np.sqrt(np.mean(residual ** 2))
        results[n] = {
            'rmse':rmse,
            'residual': residual
        }

        print(f"Window Size:{n:11s} | residualmad score: {rmse:.10f}")

    best_window = min(results, key=lambda k: results[k]['rmse'])
    print(f"\n best roling window size: {best_window}\n")


Feature  tmod

Window Size:0.5_days    | residualmad score: 0.1157858955
Window Size:1_days      | residualmad score: 0.1230634458
Window Size:7_days      | residualmad score: 0.1350407227
Window Size:14_days     | residualmad score: 0.1388845159
Window Size:21_days     | residualmad score: 0.1412109250
Window Size:30_days     | residualmad score: 0.1434955485
Window Size:60_days     | residualmad score: 0.1515938198
Window Size:90_days     | residualmad score: 0.1557816421
Window Size:120_days    | residualmad score: 0.1576805132
Window Size:150_days    | residualmad score: 0.1587745419
Window Size:300_days    | residualmad score: 0.1608514020
Window Size:360_days    | residualmad score: 0.1611995614
Window Size:720_days    | residualmad score: 0.1618623173

 best roling window size: 0.5_days

Feature  tamb

Window Size:0.5_days    | residualmad score: 1.9151820277
Window Size:1_days      | residualmad score: 2.0038563908
Window Size:7_days      | residualmad score: 2.1637097744
Windo

In [14]:
data = raw_data.copy()
data = data.drop(data[data['year']==2026].index)
data.tail()

,date,time,type,sample,standard,port,ht,tmod,tamb,lab_temp,pflow,psamp,ploop,pamb,CH4_rt,CH4_w,CH4_ht,CH4_area,CH4_skew,CH4_start_time,CH4_end_time,CH4_start_level,CH4_end_level,CH4_Rl_ht,CH4_Rl_a,CH4_Rl,CH4_norm_ht,CH4_norm_a,CH4_norm_w,CH4_R_ht,CH4_R_a,CH4_R,CH4_C_ht,CH4_C_a,CH4_C,CH4_std_stdev,CH4_std_rep,CH4_Cstd,CH4_flag_ht,CH4_flag_a,CH4_flag,CH4_flag_p,CH4_flag_ht_bin,CH4_flag_a_bin,label1,datetime_str,datetime,is_ht_999,is_area_999,is_rt_zero,is_w_zero,duration,is_duration_spike,baseline_drift,ht_area_ratio,last_std_time,seconds_since_last_std,type_std,type_air,type_tank,type_blank,type_cal,type_other,last_cal_time,seconds_since_last_cal,year,month,day,hour,minute,seconds,psamp_roll_mean_10,pamb_roll_mean_10,pflow_roll_mean_10,tmod_roll_mean_10,psamp_roll_mean_100,pamb_roll_mean_100,pflow_roll_mean_100,tmod_roll_mean_100,psamp_roll_std_10,pamb_roll_std_10,pflow_roll_std_10,CH4_rt_roll_std_10,psamp_roll_std_100,pamb_roll_std_100,pflow_roll_std_100,CH4_rt_roll_std_100,psamp_diff_1,pflow_diff_1,CH4_area_diff_1,psamp_lag_1,pflow_lag_1
808206,251231,223800,std,J-286,J-286,3,0.0,40.0,21.01,NaN,817.8,765.45,NaN,765.7,94.0,7.83,170066.0,1489856.0,1.16,80.8,133.4,23823.0,24073.0,1.00099,1.00067,1.00067,1.0010,1.0007,0.9997,1.00099,1.00067,1.00067,2017.92,2017.27,2017.27,2.406,NaN,2015.923,,,A,,0,0,0,251231223800,2025-12-31 22:38:00,False,False,False,False,52.6,False,250.0,0.114149,2025-12-31 21:57:00,2460.0,1,0,0,0,0,0,2025-12-28 06:16:00,318120.0,2025,12,31,22,38,0,766.407,766.33,816.44,40.0,772.0808,772.056,822.436,40.0,0.583306,0.586989,2.761320,0.10328,2.832881,2.881439,3.691583,0.094516,-0.47,-4.6,34189.0,765.37,813.7
808207,251231,225800,air,9m,J-286,1,9.0,40.0,20.74,NaN,813.2,765.28,NaN,765.2,94.0,7.85,171821.0,1508117.0,1.16,80.8,133.4,23750.0,24058.0,1.01178,1.01300,1.01300,1.0118,1.0130,1.0014,1.01178,1.01300,1.01300,2039.67,2042.12,2042.12,2.396,2.427,2015.923,,,A,,0,0,0,251231225800,2025-12-31 22:58:00,False,False,False,False,52.6,False,308.0,0.113931,2025-12-31 21:57:00,3660.0,0,1,0,0,0,0,2025-12-28 06:16:00,319320.0,2025,12,31,22,58,0,766.222,766.18,816.24,40.0,771.9890,771.966,822.381,40.0,0.561581,0.528730,2.555691,0.10328,2.897443,2.938020,3.719446,0.095219,0.08,4.1,-34481.0,765.45,817.8
808208,251231,231800,std,J-286,J-286,3,0.0,40.0,20.88,NaN,817.5,765.08,NaN,764.9,94.2,7.84,169569.0,1487622.0,1.06,80.8,133.6,23740.0,24030.0,0.99744,0.99876,0.99876,0.9973,0.9986,1.0015,0.99744,0.99876,0.99876,2010.76,2013.42,2013.42,2.413,NaN,2015.923,,,A,,0,0,0,251231231800,2025-12-31 23:18:00,False,False,False,False,52.8,False,290.0,0.113987,2025-12-31 22:38:00,2400.0,1,0,0,0,0,0,2025-12-28 06:16:00,320520.0,2025,12,31,23,18,0,766.039,766.00,816.19,40.0,771.8964,771.871,822.240,40.0,0.537720,0.524934,2.615106,0.10328,2.962336,3.001626,3.797527,0.095874,-0.17,-4.6,18261.0,765.28,813.2
808209,251231,233800,air,9m,J-286,1,9.0,40.0,20.79,NaN,813.3,765.05,NaN,765.0,94.2,7.86,170281.0,1496142.0,1.07,80.8,133.6,23756.0,24015.0,1.00289,1.00503,1.00503,1.0030,1.0051,1.0021,1.00289,1.00503,1.00503,2021.75,2026.06,2026.06,2.401,2.413,2015.923,,,A,,0,0,0,251231233800,2025-12-31 23:38:00,False,False,False,False,52.8,False,259.0,0.113813,2025-12-31 22:38:00,3600.0,0,1,0,0,0,0,2025-12-28 06:16:00,321720.0,2025,12,31,23,38,0,765.872,765.82,816.02,40.0,771.8028,771.775,822.179,40.0,0.551519,0.565292,2.447584,0.10328,3.028294,3.069445,3.824358,0.095874,-0.20,4.3,-20495.0,765.08,817.5
808210,251231,235800,std,J-286,J-286,3,0.0,NaN,NaN,NaN,817.4,764.90,NaN,NaN,94.2,7.83,169985.0,1489453.0,1.09,80.8,133.4,23691.0,24015.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.415,NaN,2015.923,,,A,,0,0,0,251231235800,2025-12-31 23:58:00,False,False,False,False,52.6,False,324.0,0.114126,2025-12-31 23:18:00,2400.0,1,0,0,0,0,0,2025-12-28 06:16:00,322920.0,2025,12,31,23,58,0,765.710,765.66,815.82,40.0,771.7077,771.680,822.040,40.0,0.528520,0.546097,2.590495,0.10328,3.089538,3.130656,3.892028,0.095874,-0.03,-4.2,8520.0,765.05,813.3


In [15]:
def types_convert(type_name):
    t = str(type_name).lower().strip()

    if 'cal' in t:
        return 'cal'
    elif 'std' in t:
        return 'std'
    elif 'air' == t:
        return 'air'
    elif 'tank' in t:
        return 'tank'
    elif 'blank' in t:
        return 'blank'
    else:
        return 'other'
    

In [16]:
data['type_norm'] = data['type'].apply(types_convert)
data['type_norm'].unique()

array(['air', 'std', 'tank', 'cal', 'blank', 'other'], dtype=object)

In [17]:
data = data.sort_values('datetime').reset_index(drop=True)
data.head()

,date,time,type,sample,standard,port,ht,tmod,tamb,lab_temp,pflow,psamp,ploop,pamb,CH4_rt,CH4_w,CH4_ht,CH4_area,CH4_skew,CH4_start_time,CH4_end_time,CH4_start_level,CH4_end_level,CH4_Rl_ht,CH4_Rl_a,CH4_Rl,CH4_norm_ht,CH4_norm_a,CH4_norm_w,CH4_R_ht,CH4_R_a,CH4_R,CH4_C_ht,CH4_C_a,CH4_C,CH4_std_stdev,CH4_std_rep,CH4_Cstd,CH4_flag_ht,CH4_flag_a,CH4_flag,CH4_flag_p,CH4_flag_ht_bin,CH4_flag_a_bin,label1,datetime_str,datetime,is_ht_999,is_area_999,is_rt_zero,is_w_zero,duration,is_duration_spike,baseline_drift,ht_area_ratio,last_std_time,seconds_since_last_std,type_std,type_air,type_tank,type_blank,type_cal,type_other,last_cal_time,seconds_since_last_cal,year,month,day,hour,minute,seconds,psamp_roll_mean_10,pamb_roll_mean_10,pflow_roll_mean_10,tmod_roll_mean_10,psamp_roll_mean_100,pamb_roll_mean_100,pflow_roll_mean_100,tmod_roll_mean_100,psamp_roll_std_10,pamb_roll_std_10,pflow_roll_std_10,CH4_rt_roll_std_10,psamp_roll_std_100,pamb_roll_std_100,pflow_roll_std_100,CH4_rt_roll_std_100,psamp_diff_1,pflow_diff_1,CH4_area_diff_1,psamp_lag_1,pflow_lag_1,type_norm
0,940216,142900,air,10m,G-024,3,10.0,40.00,32.27,NaN,841.9,761.10,NaN,761.1,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,F,F,,,1,1,1,940216142900,1994-02-16 14:29:00,False,False,True,True,0.0,False,0.0,NaN,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,14,29,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,air
1,940216,144900,air,10m,G-024,3,10.0,40.00,32.48,NaN,840.4,761.07,NaN,761.2,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,F,F,,,1,1,1,940216144900,1994-02-16 14:49:00,False,False,True,True,0.0,False,0.0,NaN,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,14,49,0,761.100000,761.100000,841.900000,40.000000,761.100000,761.100000,841.900000,40.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,761.10,841.9,air
2,940216,150900,air,10m,G-024,3,10.0,40.01,32.75,NaN,839.4,760.99,NaN,760.9,109.8,2.60,190908.0,496440.0,0.63,102.0,135.4,10063.0,10029.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,F,F,,,1,1,1,940216150900,1994-02-16 15:09:00,False,False,False,False,33.4,False,-34.0,0.384554,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,15,9,0,761.085000,761.150000,841.150000,40.000000,761.085000,761.150000,841.150000,40.000000,0.021213,0.070711,1.060660,0.000000,0.021213,0.070711,1.060660,0.000000,-0.03,-1.5,0.0,761.07,840.4,air
3,940216,152900,air,10m,G-024,3,10.0,40.00,32.92,NaN,839.6,760.96,NaN,760.9,94.8,6.85,151667.0,1065982.0,1.02,86.2,129.0,99999.0,99999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,,,,,0,0,0,940216152900,1994-02-16 15:29:00,False,False,False,False,42.8,False,0.0,0.142279,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,15,29,0,761.053333,761.066667,840.566667,40.003333,761.053333,761.066667,840.566667,40.003333,0.056862,0.152753,1.258306,63.393060,0.056862,0.152753,1.258306,63.393060,-0.08,-1.0,496440.0,760.99,839.4,air
4,940216,180000,air,10m,G-024,3,10.0,40.00,33.31,NaN,845.4,760.72,NaN,760.6,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,F,F,,,1,1,1,940216180000,1994-02-16 18:00:00,False,False,True,True,0.0,False,0.0,NaN,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,18,0,0,761.030000,761.025000,840.325000,40.002500,761.030000,761.025000,840.325000,40.002500,0.065828,0.150000,1.135415,59.379542,0.065828,0.150000,1.135415,59.379542,-0.03,0.2,569542.0,760.96,839.6,air


In [18]:
data['datetime'] = pd.to_datetime(data['datetime'])
data['time_diff'] = data['datetime'].diff()
data.head()

,date,time,type,sample,standard,port,ht,tmod,tamb,lab_temp,pflow,psamp,ploop,pamb,CH4_rt,CH4_w,CH4_ht,CH4_area,CH4_skew,CH4_start_time,CH4_end_time,CH4_start_level,CH4_end_level,CH4_Rl_ht,CH4_Rl_a,CH4_Rl,CH4_norm_ht,CH4_norm_a,CH4_norm_w,CH4_R_ht,CH4_R_a,CH4_R,CH4_C_ht,CH4_C_a,CH4_C,CH4_std_stdev,CH4_std_rep,CH4_Cstd,CH4_flag_ht,CH4_flag_a,CH4_flag,CH4_flag_p,CH4_flag_ht_bin,CH4_flag_a_bin,label1,datetime_str,datetime,is_ht_999,is_area_999,is_rt_zero,is_w_zero,duration,is_duration_spike,baseline_drift,ht_area_ratio,last_std_time,seconds_since_last_std,type_std,type_air,type_tank,type_blank,type_cal,type_other,last_cal_time,seconds_since_last_cal,year,month,day,hour,minute,seconds,psamp_roll_mean_10,pamb_roll_mean_10,pflow_roll_mean_10,tmod_roll_mean_10,psamp_roll_mean_100,pamb_roll_mean_100,pflow_roll_mean_100,tmod_roll_mean_100,psamp_roll_std_10,pamb_roll_std_10,pflow_roll_std_10,CH4_rt_roll_std_10,psamp_roll_std_100,pamb_roll_std_100,pflow_roll_std_100,CH4_rt_roll_std_100,psamp_diff_1,pflow_diff_1,CH4_area_diff_1,psamp_lag_1,pflow_lag_1,type_norm,time_diff
0,940216,142900,air,10m,G-024,3,10.0,40.00,32.27,NaN,841.9,761.10,NaN,761.1,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,F,F,,,1,1,1,940216142900,1994-02-16 14:29:00,False,False,True,True,0.0,False,0.0,NaN,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,14,29,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,air,NaT
1,940216,144900,air,10m,G-024,3,10.0,40.00,32.48,NaN,840.4,761.07,NaN,761.2,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,F,F,,,1,1,1,940216144900,1994-02-16 14:49:00,False,False,True,True,0.0,False,0.0,NaN,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,14,49,0,761.100000,761.100000,841.900000,40.000000,761.100000,761.100000,841.900000,40.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,761.10,841.9,air,0 days 00:20:00
2,940216,150900,air,10m,G-024,3,10.0,40.01,32.75,NaN,839.4,760.99,NaN,760.9,109.8,2.60,190908.0,496440.0,0.63,102.0,135.4,10063.0,10029.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,F,F,,,1,1,1,940216150900,1994-02-16 15:09:00,False,False,False,False,33.4,False,-34.0,0.384554,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,15,9,0,761.085000,761.150000,841.150000,40.000000,761.085000,761.150000,841.150000,40.000000,0.021213,0.070711,1.060660,0.000000,0.021213,0.070711,1.060660,0.000000,-0.03,-1.5,0.0,761.07,840.4,air,0 days 00:20:00
3,940216,152900,air,10m,G-024,3,10.0,40.00,32.92,NaN,839.6,760.96,NaN,760.9,94.8,6.85,151667.0,1065982.0,1.02,86.2,129.0,99999.0,99999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,,,,,0,0,0,940216152900,1994-02-16 15:29:00,False,False,False,False,42.8,False,0.0,0.142279,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,15,29,0,761.053333,761.066667,840.566667,40.003333,761.053333,761.066667,840.566667,40.003333,0.056862,0.152753,1.258306,63.393060,0.056862,0.152753,1.258306,63.393060,-0.08,-1.0,496440.0,760.99,839.4,air,0 days 00:20:00
4,940216,180000,air,10m,G-024,3,10.0,40.00,33.31,NaN,845.4,760.72,NaN,760.6,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1692.489,F,F,,,1,1,1,940216180000,1994-02-16 18:00:00,False,False,True,True,0.0,False,0.0,NaN,NaN,NaN,0,1,0,0,0,0,NaN,NaN,1994,2,16,18,0,0,761.030000,761.025000,840.325000,40.002500,761.030000,761.025000,840.325000,40.002500,0.065828,0.150000,1.135415,59.379542,0.065828,0.150000,1.135415,59.379542,-0.03,0.2,569542.0,760.96,839.6,air,0 days 02:31:00


In [19]:
data['session_id'] = (data['time_diff'] > pd.Timedelta(days=7)).cumsum()
data.tail()

,date,time,type,sample,standard,port,ht,tmod,tamb,lab_temp,pflow,psamp,ploop,pamb,CH4_rt,CH4_w,CH4_ht,CH4_area,CH4_skew,CH4_start_time,CH4_end_time,CH4_start_level,CH4_end_level,CH4_Rl_ht,CH4_Rl_a,CH4_Rl,CH4_norm_ht,CH4_norm_a,CH4_norm_w,CH4_R_ht,CH4_R_a,CH4_R,CH4_C_ht,CH4_C_a,CH4_C,CH4_std_stdev,CH4_std_rep,CH4_Cstd,CH4_flag_ht,CH4_flag_a,CH4_flag,CH4_flag_p,CH4_flag_ht_bin,CH4_flag_a_bin,label1,datetime_str,datetime,is_ht_999,is_area_999,is_rt_zero,is_w_zero,duration,is_duration_spike,baseline_drift,ht_area_ratio,last_std_time,seconds_since_last_std,type_std,type_air,type_tank,type_blank,type_cal,type_other,last_cal_time,seconds_since_last_cal,year,month,day,hour,minute,seconds,psamp_roll_mean_10,pamb_roll_mean_10,pflow_roll_mean_10,tmod_roll_mean_10,psamp_roll_mean_100,pamb_roll_mean_100,pflow_roll_mean_100,tmod_roll_mean_100,psamp_roll_std_10,pamb_roll_std_10,pflow_roll_std_10,CH4_rt_roll_std_10,psamp_roll_std_100,pamb_roll_std_100,pflow_roll_std_100,CH4_rt_roll_std_100,psamp_diff_1,pflow_diff_1,CH4_area_diff_1,psamp_lag_1,pflow_lag_1,type_norm,time_diff,session_id
808206,251231,223800,std,J-286,J-286,3,0.0,40.0,21.01,NaN,817.8,765.45,NaN,765.7,94.0,7.83,170066.0,1489856.0,1.16,80.8,133.4,23823.0,24073.0,1.00099,1.00067,1.00067,1.0010,1.0007,0.9997,1.00099,1.00067,1.00067,2017.92,2017.27,2017.27,2.406,NaN,2015.923,,,A,,0,0,0,251231223800,2025-12-31 22:38:00,False,False,False,False,52.6,False,250.0,0.114149,2025-12-31 21:57:00,2460.0,1,0,0,0,0,0,2025-12-28 06:16:00,318120.0,2025,12,31,22,38,0,766.407,766.33,816.44,40.0,772.0808,772.056,822.436,40.0,0.583306,0.586989,2.761320,0.10328,2.832881,2.881439,3.691583,0.094516,-0.47,-4.6,34189.0,765.37,813.7,std,0 days 00:20:00,5
808207,251231,225800,air,9m,J-286,1,9.0,40.0,20.74,NaN,813.2,765.28,NaN,765.2,94.0,7.85,171821.0,1508117.0,1.16,80.8,133.4,23750.0,24058.0,1.01178,1.01300,1.01300,1.0118,1.0130,1.0014,1.01178,1.01300,1.01300,2039.67,2042.12,2042.12,2.396,2.427,2015.923,,,A,,0,0,0,251231225800,2025-12-31 22:58:00,False,False,False,False,52.6,False,308.0,0.113931,2025-12-31 21:57:00,3660.0,0,1,0,0,0,0,2025-12-28 06:16:00,319320.0,2025,12,31,22,58,0,766.222,766.18,816.24,40.0,771.9890,771.966,822.381,40.0,0.561581,0.528730,2.555691,0.10328,2.897443,2.938020,3.719446,0.095219,0.08,4.1,-34481.0,765.45,817.8,air,0 days 00:20:00,5
808208,251231,231800,std,J-286,J-286,3,0.0,40.0,20.88,NaN,817.5,765.08,NaN,764.9,94.2,7.84,169569.0,1487622.0,1.06,80.8,133.6,23740.0,24030.0,0.99744,0.99876,0.99876,0.9973,0.9986,1.0015,0.99744,0.99876,0.99876,2010.76,2013.42,2013.42,2.413,NaN,2015.923,,,A,,0,0,0,251231231800,2025-12-31 23:18:00,False,False,False,False,52.8,False,290.0,0.113987,2025-12-31 22:38:00,2400.0,1,0,0,0,0,0,2025-12-28 06:16:00,320520.0,2025,12,31,23,18,0,766.039,766.00,816.19,40.0,771.8964,771.871,822.240,40.0,0.537720,0.524934,2.615106,0.10328,2.962336,3.001626,3.797527,0.095874,-0.17,-4.6,18261.0,765.28,813.2,std,0 days 00:20:00,5
808209,251231,233800,air,9m,J-286,1,9.0,40.0,20.79,NaN,813.3,765.05,NaN,765.0,94.2,7.86,170281.0,1496142.0,1.07,80.8,133.6,23756.0,24015.0,1.00289,1.00503,1.00503,1.0030,1.0051,1.0021,1.00289,1.00503,1.00503,2021.75,2026.06,2026.06,2.401,2.413,2015.923,,,A,,0,0,0,251231233800,2025-12-31 23:38:00,False,False,False,False,52.8,False,259.0,0.113813,2025-12-31 22:38:00,3600.0,0,1,0,0,0,0,2025-12-28 06:16:00,321720.0,2025,12,31,23,38,0,765.872,765.82,816.02,40.0,771.8028,771.775,822.179,40.0,0.551519,0.565292,2.447584,0.10328,3.028294,3.069445,3.824358,0.095874,-0.20,4.3,-20495.0,765.08,817.5,air,0 days 00:20:00,5
808210,251231,235800,std,J-286,J-286,3,0.0,NaN,NaN,NaN,817.4,764.90,NaN,NaN,94.2,7.83,169985.0,1489453.0,1.09,80.8,133.4,23691.0,24015.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.415,NaN,2015.923,,,A,,0,0,0,251231235800,2025-12-31 23:58:00,False,False,False,False,52.6,False,324.0,0.114126,2025-12-31 23:18:00,2400.0,1,0,0,0,0,0,2025-12-28 06:16:00,322920.0,2025,12,31,23,58,0,765.710,765.66,815.82,40.0,771.7077,771.680,822.040,40.0,0.52852

In [20]:
final_window = 36 * 7
for col in feature_cols :
    data[f'{col}_drift_7_days'] = data.groupby(['session_id', 'type_norm'])[col].transform(
        lambda x: x.rolling(window=final_window, min_periods=1, center=False).mean())
    data[f'{col}_residual'] = data[col] - data[f'{col}_drift_7_days']


In [21]:
data = pd.get_dummies(data, columns=['type_norm'], drop_first=False, dtype=float)
print(data.columns)

Index(['date', 'time', 'type', 'sample', 'standard', 'port', 'ht', 'tmod',
       'tamb', 'lab_temp',
       ...
       'duration_drift_7_days', 'duration_residual',
       'ht_area_ratio_drift_7_days', 'ht_area_ratio_residual', 'type_norm_air',
       'type_norm_blank', 'type_norm_cal', 'type_norm_other', 'type_norm_std',
       'type_norm_tank'],
      dtype='object', length=132)


In [22]:
feature_cols

['tmod',
 'tamb',
 'pflow',
 'psamp',
 'pamb',
 'CH4_rt',
 'CH4_w',
 'CH4_ht',
 'CH4_area',
 'CH4_skew',
 'CH4_start_time',
 'CH4_end_time',
 'CH4_start_level',
 'CH4_end_level',
 'duration',
 'ht_area_ratio']

In [23]:
raw_features = feature_cols

res_features = [f'{col}_residual' for col in raw_features]

type_features = ['type_norm_air',
       'type_norm_blank', 'type_norm_cal', 'type_norm_other', 'type_norm_std',
       'type_norm_tank']

all_features = raw_features + res_features + type_features

print(f"Feature sletected:{len(all_features)}")


Feature sletected:38


In [24]:
all_features

['tmod',
 'tamb',
 'pflow',
 'psamp',
 'pamb',
 'CH4_rt',
 'CH4_w',
 'CH4_ht',
 'CH4_area',
 'CH4_skew',
 'CH4_start_time',
 'CH4_end_time',
 'CH4_start_level',
 'CH4_end_level',
 'duration',
 'ht_area_ratio',
 'tmod_residual',
 'tamb_residual',
 'pflow_residual',
 'psamp_residual',
 'pamb_residual',
 'CH4_rt_residual',
 'CH4_w_residual',
 'CH4_ht_residual',
 'CH4_area_residual',
 'CH4_skew_residual',
 'CH4_start_time_residual',
 'CH4_end_time_residual',
 'CH4_start_level_residual',
 'CH4_end_level_residual',
 'duration_residual',
 'ht_area_ratio_residual',
 'type_norm_air',
 'type_norm_blank',
 'type_norm_cal',
 'type_norm_other',
 'type_norm_std',
 'type_norm_tank']

In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
data[all_features] = scaler.fit_transform(data[all_features].astype(float))

In [27]:
import torch
df_normal = data[data['label1'] == 0].copy()

X_train = torch.tensor(df_normal[all_features].values, dtype=torch.float32)

X_all = torch.tensor(data[all_features].values, dtype=torch.float32)
y_all = data['label1'].values

print(f"Training shape:{X_train.shape}")
print(f"All data shape:{X_all.shape}")

Training shape:torch.Size([769380, 38])
All data shape:torch.Size([808211, 38])


In [29]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 512

train_dataset = TensorDataset(X_train)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"BATCH {len(train_loader)}")

BATCH 1503


In [ ]:
import torch.nn as nn
class GCAutoencoder(nn.Module):
    def __init__(self, input_dim=26):
        super(GCAutoencoder, self).__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8,3)
        )

        self.decoder = nn.Sequential(
            nn.Linear(3,8),
            nn.ReLU(),
            nn.Linear(8,16),
            nn.RELU(),
            nn.Linear(16, input_dim)
        )

        def forward(self, x):
            embedding = self.encoder(x)

            reconstructed = self.decoder(embedding)
            return reconstructed, embedding
        
model = GCAutoencoder(input_dim=38)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)



In [ ]:
import torch.nn as nn

model = GCAutoencoder(input_dim=38)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [ ]:
import time

EPOCHS = 30
loss_history = []

print("Training ...")

model.train()

for epoch in range(EPOCHS):
    start_time = time.time()
    running_loss  = 0.0

    for batch in train_loader:
        inputs = batch[0].to(device)

        optimizer.zero_grad()

        reconstructed, embedding = model(inputs)

        loss = criterion(reconstructed, inputs)

        loss.backward()
        optimizer.step()

        running_loss +=loss.item() * inputs.size(0)
    epoch_loss = running_loss / len(X_train)
    loss_history = []
    loss_history.append(epoch_loss)

    elapsed = time.time() - start_time()
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | LOSS: {epoch_loss:.6f} | TIME COMSUME: {elapsed:.2f} s")
print("Finished")



In [ ]:
torch.save(model.state_dict(), 'gc_autoencoder_model.pth')

import joblib
joblib.dump(scaler, 'data_scaler.pkl')
print("SUCCESSED!")


In [ ]:
model.eval()
with torch.no_grad():

    X_all_device = X_all.to(device)
    reconstructed_all, embedding_all = model(X_all_device)

    mse_per_row = torch.mean((reconstructed_all - X_all_device) ** 2, dim=1).cpu().numpy()

data['anomaly_score'] = mse_per_row


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

emb_3d = embedding_all.cpu().numpy()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplots(111, projection='3d')


scatter = ax.scatter(emb_3d[:, 0], emb_3d[:,1], emb_3d[:, 2],
                     c=data['datetime'].dt.year, cmap='jet',s = 4,alpha=0.6)

cbar = plt.colorbar(scatter)
cbar.set_label('Year 1994 - 2025')
ax.set_title('Data Convergence in Latent Space (3D Embedding)')

plt.show()

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(data['datetime'], data['anomaly_score'], label='Anomaly Score(MSE)', color='blue', alpha=0.7)

plt.fill_between(data['datetime'], 0, data['anomaly_score'].max(),
                 where = (data['label1'] == 1), color = 'red', label="T F Period")

threshold = np.percentitle(data[data['label1'] == 0]['anomaly_score'], 99)
plt.axhline(y-threshold, color='green', linestyle='--', label=f'Threshold (99 th percent: {threshold:.4f})')

plt.title('Real-Time Anomaly Detection PerformanceOver 30 Years')
plt.xlabel('Timeline')
plt.ylabel('Reconstruction Error (MSE)')
plt.legend()
plt.show()